# Практична робота 20. Лінійне програмування
**Варіант 1 (м. Київ)**

Цей ноутбук містить повний розв'язок задачі лінійного програмування для виробничої дільниці: математичну постановку (у коментарях), оптимізацію через `scipy.optimize.linprog`, аналіз активних обмежень (вузьких місць) та сценарний аналіз What-if.

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import linprog

# Прибуток:
profit_A = 60    # грн/од
profit_B = 45    

# Ресурси (Виріб А, Виріб Б, Ліміт):
# 1. Сировина (кг)
raw_A = 3; raw_B = 5; raw_limit = 300
# 2. Робочий час (год)
time_A = 4; time_B = 2; time_limit = 200
# 3. Електроенергія (кВт·год)
elec_A = 1; elec_B = 1; elec_limit = 95

## Завдання 1. Математична постановка власного варіанта
*(Поставлено повністю у вигляді коментарів перед кодом оптимізації)*

In [2]:
"""
ЗАВДАННЯ 1. МАТЕМАТИЧНА ПОСТАНОВКА ЗАДАЧІ ЛІНІЙНОГО ПРОГРАМУВАННЯ (Варіант 1 — Київ)

1. Змінні decision variables:
   - x1 — кількість виробу А, що випускається на тиждень (од.)
   - x2 — кількість виробу Б, що випускається на тиждень (од.)

2. Цільова функція (Максимізація тижневого прибутку):
   Максимізувати Z = 60 * x1 + 45 * x2
   (Оскільки linprog мінімізує, у код передаємо з мінусом: c = [-60, -45])

3. Система обмежень (Ресурси виробництва):
   - Обмеження по сировині:      3 * x1 + 5 * x2 <= 300
   - Обмеження по машинному часу: 4 * x1 + 2 * x2 <= 200
   - Обмеження по електроенергії: 1 * x1 + 1 * x2 <= 95
   - Умова невід'ємності:        x1 >= 0, x2 >= 0
"""
print("Математична постановка успішно задокументована у коментарях.")

Математична постановка успішно задокументована у коментарях.


## Завдання 2 & 3. Розв'язання через linprog та Інтерпретація оптимального розв'язку

In [ ]:

c = [-profit_A, -profit_B]

# Матриця нерівностей формату A_ub @ x <= b_ub
A_ub = [
    [raw_A, raw_B],   # Сировина: 3*x1 + 5*x2 <= 300
    [time_A, time_B], # Час:      4*x1 + 2*x2 <= 200
    [elec_A, elec_B]  # Електро:  1*x1 + 1*x2 <= 95
]
b_ub = [raw_limit, time_limit, elec_limit]

# Межі невід'ємності для x1 та x2
bounds = [(0, None), (0, None)]

# Розв'язання задачі ЛП методом 'highs'
res = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method="highs")

print("Статус оптимізації:", res.status, "-", res.message)
if res.status == 0:
    opt_x1, opt_x2 = res.x
    max_profit = -res.fun
    print(f"\nОптимальний випуск:")
    print(f"- Виріб А (x1): {opt_x1:.2f} од./тиждень")
    print(f"- Виріб Б (x2): {opt_x2:.2f} од./тиждень")
    print(f"Максимальний тижневий прибуток: {max_profit:.2f} грн")
"""
ЗАВДАННЯ 3. ІНТЕРПРЕТАЦІЯ ОПТИМАЛЬНОГО РОЗВ'ЯЗКУ
- Оптимальний план передбачає випуск певного обсягу виріб А та Б.
- Оскільки linprog видає неперервні (не цілі) числа, в реальному виробництві їх доведеться округлити.
- Проте просте округлення не завжди дає гарантовано оптимальний цілочисловий розв'язок (через ризик виходу за межі обмежень або недоотримання прибутку), тому для точного цілочислового випуску застосовують спеціальні методи ILP (цілочислового лінійного програмування).
"""

Статус оптимізації: 0 - Optimization terminated successfully. (HiGHS Status 7: Optimal)

Оптимальний випуск:
- Виріб А (x1): 28.57 од./тиждень
- Виріб Б (x2): 42.86 од./тиждень
Максимальний тижневий прибуток: 3642.86 грн


"\nЗАВДАННЯ 3. ІНТЕРПРЕТАЦІЯ ОПТИМАЛЬНОГО РОЗВ'ЯЗКУ\n- Оптимальний план передбачає випуск певного обсягу виріб А та Б.\n- Оскільки linprog видає неперервні (не цілі) числа, в реальному виробництві їх доведеться округлити.\n- Проте просте округлення не завжди дає гарантовано оптимальний цілочисловий розв'язок (через ризик виходу за межі обмежень або недоотримання прибутку), тому для точного цілочислового випуску застосовують спеціальні методи ILP (цілочислового лінійного програмування).\n"

## Завдання 4. Активні обмеження (Вузькі місця)

In [ ]:

slack_values = res.slack
resources_names = ["Сировина (кг)", "Робочий час (год)", "Електроенергія (кВт·год)"]

print("Аналіз використання ресурсів (slack):")
for name, slk, limit in zip(resources_names, slack_values, b_ub):
    used = limit - slk
    is_active = "АКТИВНЕ (Вузьке місце)" if abs(slk) < 1e-5 else f"Має запас ({slk:.2f})"
    print(f"- {name}: використано {used:.2f} з {limit} | Статус: {is_active}")

Аналіз використання ресурсів (slack):
- Сировина (кг): використано 300.00 з 300 | Статус: АКТИВНЕ (Вузьке місце)
- Робочий час (год): використано 200.00 з 200 | Статус: АКТИВНЕ (Вузьке місце)
- Електроенергія (кВт·год): використано 71.43 з 95 | Статус: Має запас (23.57)


## Завдання 5. What-if: зміна ресурсних обмежень

In [ ]:

active_idx = np.argmin(np.abs(res.slack))
b_ub_modified_active = b_ub.copy()
b_ub_modified_active[active_idx] *= 1.15  

res_active = linprog(c, A_ub=A_ub, b_ub=b_ub_modified_active, bounds=bounds, method="highs")
print(f"При збільшенні активного ресурсу на 15%:")
print(f"- Новий прибуток: {-res_active.fun:.2f} грн (зріс з {max_profit:.2f})")

inactive_idx = np.argmax(res.slack)
b_ub_modified_inactive = b_ub.copy()
b_ub_modified_inactive[inactive_idx] *= 1.50

res_inactive = linprog(c, A_ub=A_ub, b_ub=b_ub_modified_inactive, bounds=bounds, method="highs")
print(f"\nПри збільшенні неактивного ресурсу на 50%:")
print(f"- Новий прибуток: {-res_inactive.fun:.2f} грн (залишився без змін)")

При збільшенні активного ресурсу на 15%:
- Новий прибуток: 3835.71 грн (зріс з 3642.86)

При збільшенні неактивного ресурсу на 50%:
- Новий прибуток: 3642.86 грн (залишився без змін)
